# wrap-forward-fn-generic composite — cx13: wrap_forward_fn threads kwargs into call AND Recipe

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `kwargs-pass-through-recipe`, `wrap-forward-fn-generic`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "wrap-forward-fn-generic"
DD_ATOM_IDS = ["kwargs-pass-through-recipe", "wrap-forward-fn-generic"]
DD_SUBTOPICS = ["Backprop: Kwargs pass-through", "Backprop: wrap forward fn"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing wrap_forward_fn with kwargs-pass-through Recipe

`wrap_forward_fn` is the factory that converts a raw numerical fn into a `MiniTensor`-aware wrapper. The unbox/call/box skeleton is the same for every op, but ops with **keyword args** (`sum(x, dim=1, keepdim=True)`) force TWO places where the kwargs must thread:

1. into the **forward call** — otherwise `sum` reduces the wrong axis.
2. onto the **Recipe** — otherwise the reverse pass has no way to call `back_fn(..., **recipe.kwargs)` with the same kwargs.

This composite exercises both atoms in one wrapper: write `wrap_forward_fn(fwd_fn)` so it (a) unboxes MiniTensor inputs, (b) calls `fwd_fn(*raw_args, **kwargs)` with kwargs threaded, and (c) attaches `Recipe(func, raw_args, kwargs, parents)` carrying the SAME kwargs dict the call used.

The forward result is observable. The Recipe-kwargs storage is INVISIBLE until the reverse pass runs — which is exactly when the bug bites. The test checks both directly.

### Composite Exercise — wrap_forward_fn threads kwargs into call AND Recipe

**Atoms exercised together**: `kwargs-pass-through-recipe`, `wrap-forward-fn-generic`

Write `cx13_wrap_forward_fn(fwd_fn)` that returns a closure `tensor_func(*args, **kwargs)` which:

1. **Unboxes** every `MiniTensor` arg to its `.array`; passes non-Tensor args through unchanged.
2. Calls `fwd_fn(*raw_args, **kwargs)` — kwargs MUST reach the call so `sum(x, dim=1)` reduces the right axis.
3. Builds `out = MiniTensor(out_raw)` and attaches `out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)` where `parents = {idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)}`. The Recipe must store the SAME kwargs dict the call used.

Use `t.sum` as the witness op (it takes `dim` and `keepdim`). Don't touch the differentiability gate — assume every op is differentiable for this drill.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

from dataclasses import dataclass, field
from typing import Callable, Optional

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx13_wrap_forward_fn(fwd_fn):
    """Return tensor_func that threads kwargs into call AND Recipe."""
    raise NotImplementedError

def _test_cx13():
    wrapped_sum = cx13_wrap_forward_fn(t.sum)
    x = MiniTensor(t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]))

    # (a) forward call gets kwargs (dim=1)
    out = wrapped_sum(x, dim=1)
    assert isinstance(out, MiniTensor), 'output must be a MiniTensor'
    assert t.allclose(out.array, t.tensor([6.0, 15.0])), (
        f'forward dim=1 was ignored: {out.array} (expected [6, 15])'
    )

    # (b) Recipe carries the SAME kwargs the call used
    assert out.recipe is not None, 'Recipe was never attached'
    assert out.recipe.func is t.sum
    assert out.recipe.kwargs == {'dim': 1}, (
        f'Recipe.kwargs missing or wrong: {out.recipe.kwargs}'
    )
    assert 0 in out.recipe.parents and out.recipe.parents[0] is x

    # (c) Recipe.args are RAW unboxed tensors
    assert isinstance(out.recipe.args[0], t.Tensor)
    assert not isinstance(out.recipe.args[0], MiniTensor), 'args must be unboxed'

    # (d) two kwargs thread through together
    out2 = wrapped_sum(x, dim=1, keepdim=True)
    assert out2.array.shape == (2, 1), f'keepdim ignored: {out2.array.shape}'
    assert out2.recipe.kwargs == {'dim': 1, 'keepdim': True}

    # (e) no-kwargs case stores empty dict (NOT None)
    wrapped_log = cx13_wrap_forward_fn(t.log)
    y = MiniTensor(t.tensor([1.0, t.e, t.e * t.e]))
    out3 = wrapped_log(y)
    assert t.allclose(out3.array, t.tensor([0.0, 1.0, 2.0]), atol=1e-5)
    assert out3.recipe.kwargs == {}, (
        f'empty kwargs must be {{}}, got {out3.recipe.kwargs!r}'
    )

    # (f) the wrapper is a fresh closure per call
    wrapped_log2 = cx13_wrap_forward_fn(t.log)
    assert wrapped_log2 is not wrapped_log, 'each wrap returns a new closure'

    # (g) non-Tensor args pass through unchanged
    wrapped_clamp = cx13_wrap_forward_fn(t.clamp)
    z = MiniTensor(t.tensor([-1.0, 0.5, 2.0]))
    out4 = wrapped_clamp(z, min=0.0, max=1.0)
    assert t.allclose(out4.array, t.tensor([0.0, 0.5, 1.0]))
    assert out4.recipe.kwargs == {'min': 0.0, 'max': 1.0}
    _dd_passed.add('cx13')

_test_cx13()

<details><summary>Show solution — cx13</summary>

```python
from dataclasses import dataclass, field
from typing import Callable, Optional

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx13_wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(
            a.array if isinstance(a, MiniTensor) else a for a in args
        )
        out_raw = fwd_fn(*raw_args, **kwargs)
        parents = {
            i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)
        }
        out = MiniTensor(out_raw)
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func
```

The two places kwargs flow — `fwd_fn(*raw_args, **kwargs)` AND `Recipe(..., kwargs, ...)` — are independent: forgetting either is a silent bug that only surfaces on the reverse pass.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx13'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx13',
        'subtopics': ["Backprop: Kwargs pass-through", "Backprop: wrap forward fn"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()